# Chapter 11: Intersection in 3D

Source orientation: printed pages 481-662; PDF pages 528-709. The assigned PDF span was read with `pdftotext`; it was used only to identify the chapter structure and algorithm families. All prose, code, examples, diagrams, and checks here are original notebook material.

This chapter is a practical catalog of 3D intersection queries. The same question keeps reappearing in different clothing: choose a representation, solve for a parameter or projection interval, restrict the answer to the primitive's domain, and then classify the returned set as empty, a point, a segment, a polygon, or a curve. This notebook turns that workflow into executable examples for line/plane, ray/triangle, polygon and polyhedron slicing, quadrics, boxes/slabs, cylinder/cone/torus roots, and separating-axis diagnostics.


## Chapter Goal

Build a reusable mental model for 3D intersection algorithms used in graphics, collision detection, CAD, and mesh processing.

The core lesson is not a long list of formulas. It is the discipline of separating three layers:

- **Representation:** parametric lines, implicit planes and surfaces, barycentric triangles, half-space polyhedra, local frames for boxes and cylinders.
- **Domain filter:** a line has all real parameters, a ray keeps `t >= 0`, a segment clamps `0 <= t <= 1`, a triangle keeps nonnegative barycentric coordinates, and a finite solid keeps heights or box coordinates inside bounds.
- **Certificate:** a usable intersection result should carry residuals, intervals, barycentric weights, face labels, or projection gaps so later code can explain the answer.

A robust implementation can return `False`, but a teachable implementation returns enough state to debug why.


## Computational Translation Guide

| Chapter object | Computational representation | Intersection work | Certificate to check |
| --- | --- | --- | --- |
| Line, ray, segment | `X(t) = P + t d` plus a domain for `t` | solve a scalar, quadratic, or quartic equation in `t` | parameter range and residual on every primitive |
| Plane | unit normal `n` and offset `c` with `dot(n, X) = c` | solve one linear equation or clip signs | `abs(dot(n, Q) - c)` |
| Triangle | vertices plus barycentric coordinates | intersect supporting plane, then test weights | `u, v, w >= 0`, `u + v + w = 1` |
| Convex polyhedron | mesh vertices/faces or half-spaces | clip a line interval or slice edges by a plane | ordered section polygon and half-space residuals |
| Quadric | implicit form `F(X) = 0`, often matrix or local-frame coefficients | substitute `P + t d` to get a quadratic | roots classified by discriminant and domain |
| Cylinder/cone/torus | local-frame implicit equations | solve quadratic for cylinder/cone, quartic for torus | residual at roots plus finite-height or periodic parameter checks |
| AABB/OBB | center, axes, half-extents | slab intervals in local coordinates | `t_enter <= t_exit` and endpoints inside the box |
| Convex pair | projection intervals on candidate axes | separating-axis test over normals and edge-cross-edge axes | no positive gap means overlap; one positive gap separates |

The examples use `Plotly` for rotatable 3D scenes, `trimesh` for mesh/edge bookkeeping, `SymPy` for symbolic degree checks, `NetworkX` for the algorithm dependency map, and `Matplotlib` for durable diagnostic plots.


## Refreshed Visual Storyboard

| Visual | Library route | Learner inspection target | Validation |
| --- | --- | --- | --- |
| `storyboard-intersection-algorithm-map.png` | NetworkX + Matplotlib | how parameter solve, domain clipping, implicit residuals, and SAT fit together | graph has all planned algorithm families |
| `line-plane-ray-triangle-parameters.html` | Plotly 3D | a line-plane point and a ray-triangle hit with visible parameter and barycentric data | plane residual, ray parameter, barycentric sum |
| `polyhedron-section-slabs.html` | Trimesh + Plotly 3D | a plane section polygon of a box and the ray interval kept by slabs | section vertices satisfy the plane; slab entry precedes exit |
| `implicit-surface-root-diagnostics.png` and `implicit-surface-root-atlas.html` | NumPy polynomial arithmetic + Plotly/Matplotlib | roots as actual crossings of sphere, cylinder, cone, and torus residuals | each reported root drives the implicit residual near zero |
| `sat-axis-gap-diagnostics.png` and `sat-oriented-boxes.html` | NumPy OBB projection geometry + Matplotlib/Plotly | which candidate axis separates boxes, and which axes merely overlap | SAT decision is invariant under swapping the two boxes |

The static PNGs are durable audit artifacts. The HTML files are rotatable scenes for inspecting 3D geometry that a flat page hides.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp
import trimesh
from numpy.polynomial import Polynomial
from plotly.subplots import make_subplots

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the GTCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import artifact_report, require_nonempty

TOPIC = "chapter-11"
ENTRY_TITLE = "Intersection in 3D"
SOURCE_SPAN = {"printed": "481-662", "pdf": "528-709"}
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for folder in ["figures", "interactive", "checks"]:
    (ARTIFACT_ROOT / folder).mkdir(parents=True, exist_ok=True)

artifact_paths: list[Path] = []
check_values: dict[str, object] = {"source_span": SOURCE_SPAN}
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})


In [ ]:
def normalize(v: np.ndarray, *, eps: float = 1e-12) -> np.ndarray:
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)
    if length <= eps:
        raise ValueError(f"cannot normalize near-zero vector {v}")
    return v / length


def plane_basis(normal: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n = normalize(normal)
    seed = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.85 else np.array([0.0, 1.0, 0.0])
    u = normalize(np.cross(n, seed))
    v = normalize(np.cross(n, u))
    return u, v


def line_plane_intersection(P: np.ndarray, d: np.ndarray, n: np.ndarray, c: float, *, eps: float = 1e-12) -> dict[str, object]:
    P = np.asarray(P, dtype=float)
    d = normalize(d)
    n = normalize(n)
    denom = float(np.dot(n, d))
    signed = float(np.dot(n, P) - c)
    if abs(denom) <= eps:
        status = "coplanar" if abs(signed) <= eps else "parallel-disjoint"
        return {"status": status, "denominator": denom, "signed_origin": signed, "t": None, "point": None}
    t = -signed / denom
    Q = P + t * d
    return {"status": "point", "denominator": denom, "signed_origin": signed, "t": float(t), "point": Q, "residual": float(np.dot(n, Q) - c)}


def ray_triangle_intersection(O: np.ndarray, d: np.ndarray, tri: np.ndarray, *, eps: float = 1e-12) -> dict[str, object]:
    O = np.asarray(O, dtype=float)
    d = normalize(d)
    tri = np.asarray(tri, dtype=float)
    v0, v1, v2 = tri
    e1 = v1 - v0
    e2 = v2 - v0
    h = np.cross(d, e2)
    det = float(np.dot(e1, h))
    if abs(det) <= eps:
        return {"hit": False, "reason": "parallel", "determinant": det}
    inv_det = 1.0 / det
    s = O - v0
    u = inv_det * float(np.dot(s, h))
    q = np.cross(s, e1)
    v = inv_det * float(np.dot(d, q))
    t = inv_det * float(np.dot(e2, q))
    w = 1.0 - u - v
    hit = (t >= -eps) and (u >= -eps) and (v >= -eps) and (w >= -eps)
    Q = O + t * d
    bary_point = w * v0 + u * v1 + v * v2
    return {
        "hit": bool(hit),
        "reason": "inside" if hit else "outside-domain",
        "determinant": det,
        "t": float(t),
        "point": Q,
        "barycentric": np.array([w, u, v]),
        "point_residual": float(np.linalg.norm(Q - bary_point)),
    }


def plane_grid(normal: np.ndarray, c: float, *, width: float = 2.2, samples: int = 15) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = normalize(normal)
    u, v = plane_basis(n)
    center = c * n
    s = np.linspace(-width, width, samples)
    a, b = np.meshgrid(s, s)
    pts = center[None, None, :] + a[..., None] * u + b[..., None] * v
    return pts[..., 0], pts[..., 1], pts[..., 2]


def add_points(fig: go.Figure, points: np.ndarray, name: str, *, color: str, size: int = 6) -> None:
    pts = np.asarray(points, dtype=float)
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=name, marker={"size": size, "color": color}))


def add_polyline(fig: go.Figure, points: np.ndarray, name: str, *, color: str, width: int = 5) -> None:
    pts = np.asarray(points, dtype=float)
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines", name=name, line={"color": color, "width": width}))


## Dependency Map: One Query Pattern, Many Primitive Pairs

The chapter is broad, but the algorithms are not unrelated. A linear component starts with a parameter. A bounded primitive turns that parameter into a domain check. An implicit surface turns the same substitution into a polynomial root problem. A convex solid can be tested by clipping intervals or by projecting onto candidate separating axes. The map below is the refreshed storyboard for the notebook.


In [ ]:
G = nx.DiGraph()
edges = [
    ("primitive representation", "line parameter t"),
    ("primitive representation", "local frames"),
    ("line parameter t", "line-plane solve"),
    ("line-plane solve", "ray/triangle barycentric filter"),
    ("line-plane solve", "polygon/polyhedron clipping"),
    ("local frames", "box slab intervals"),
    ("local frames", "cylinder/cone equations"),
    ("line parameter t", "quadric residual roots"),
    ("quadric residual roots", "sphere/cylinder/cone hits"),
    ("quadric residual roots", "torus quartic hits"),
    ("polygon/polyhedron clipping", "contact polygon or segment"),
    ("box slab intervals", "entry/exit certificate"),
    ("primitive representation", "SAT candidate axes"),
    ("SAT candidate axes", "projection gap diagnostics"),
    ("projection gap diagnostics", "convex overlap certificate"),
]
G.add_edges_from(edges)
pos = {
    "primitive representation": (0.0, 0.0),
    "line parameter t": (1.5, 0.8),
    "local frames": (1.5, -0.8),
    "line-plane solve": (3.0, 1.2),
    "ray/triangle barycentric filter": (4.7, 1.8),
    "polygon/polyhedron clipping": (4.7, 0.7),
    "contact polygon or segment": (6.4, 0.7),
    "box slab intervals": (3.2, -0.3),
    "entry/exit certificate": (4.9, -0.3),
    "cylinder/cone equations": (3.2, -1.3),
    "quadric residual roots": (3.2, -2.1),
    "sphere/cylinder/cone hits": (5.0, -1.6),
    "torus quartic hits": (5.0, -2.5),
    "SAT candidate axes": (3.0, -3.3),
    "projection gap diagnostics": (4.9, -3.3),
    "convex overlap certificate": (6.6, -3.3),
}
fig, ax = plt.subplots(figsize=(11.5, 6.2))
node_colors = ["#244C5A" if n == "primitive representation" else "#6BA292" if "certificate" in n or "diagnostics" in n else "#D7B46A" for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", width=1.6, edge_color="#6B7280", connectionstyle="arc3,rad=0.03")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=1800, linewidths=1.2, edgecolors="#111827")
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_color="white")
ax.set_title("Chapter 11 storyboard: from representation to certificates", loc="left", fontsize=13, weight="bold")
ax.text(0.0, -3.85, "Inspection target: every branch ends in a residual, interval, barycentric coordinate, contact set, or projection gap.", fontsize=9, color="#374151")
ax.set_axis_off()
fig.tight_layout()
storyboard_path = save_matplotlib(fig, TOPIC, "figures", "storyboard-intersection-algorithm-map.png")
plt.close(fig)
artifact_paths.append(storyboard_path)
check_values["storyboard_nodes"] = int(G.number_of_nodes())
check_values["storyboard_edges"] = int(G.number_of_edges())
display_artifact(storyboard_path, width=900)


## Linear Components: Line/Plane and Ray/Triangle

A line-plane test is the smallest 3D intersection algorithm: substitute `P + t d` into `dot(n, X) = c` and solve one scalar equation. The denominator `dot(n, d)` distinguishes a unique point from a parallel or coplanar case. A ray or segment then adds a domain check on `t`.

A ray-triangle test adds one more domain filter. After the ray reaches the triangle's supporting plane, barycentric coordinates decide whether the point is inside the triangle. The code below uses the same ingredients a renderer would keep for debugging: the ray parameter, the determinant of the plane test, the barycentric weights, and a point reconstruction residual.


In [ ]:
plane_n = normalize(np.array([0.25, -0.35, 1.0]))
plane_c = 0.15
line_P = np.array([-1.35, -0.85, -0.75])
line_d = normalize(np.array([1.45, 0.95, 1.25]))
line_hit = line_plane_intersection(line_P, line_d, plane_n, plane_c)
line_Q = np.asarray(line_hit["point"])

tri = np.array([[0.15, -0.85, 0.10], [1.35, 0.25, 0.55], [-0.35, 0.90, 0.72]])
target_bary = np.array([0.22, 0.36, 0.42])
ray_target = target_bary @ tri
ray_d = normalize(np.array([0.22, 0.12, 1.0]))
ray_O = ray_target - 2.25 * ray_d
ray_hit = ray_triangle_intersection(ray_O, ray_d, tri)
ray_Q = np.asarray(ray_hit["point"])

X, Y, Z = plane_grid(plane_n, plane_c, width=1.55)
fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, name="plane dot(n,X)=c", opacity=0.45, showscale=False, colorscale=[[0, "#9CC9E2"], [1, "#9CC9E2"]]))
fig.add_trace(go.Mesh3d(x=tri[:, 0], y=tri[:, 1], z=tri[:, 2], i=[0], j=[1], k=[2], name="triangle domain", opacity=0.58, color="#E9B44C"))
line_ts = np.linspace(float(line_hit["t"]) - 1.25, float(line_hit["t"]) + 1.15, 40)
line_pts = line_P[None, :] + line_ts[:, None] * line_d[None, :]
add_polyline(fig, line_pts, "line P + t d", color="#1F4E79")
ray_ts = np.linspace(0.0, float(ray_hit["t"]) + 0.9, 40)
ray_pts = ray_O[None, :] + ray_ts[:, None] * ray_d[None, :]
add_polyline(fig, ray_pts, "ray, t >= 0", color="#8C2D19")
add_points(fig, np.vstack([line_Q]), f"line-plane hit t={line_hit['t']:.3f}", color="#1F4E79", size=7)
add_points(fig, np.vstack([ray_Q]), "ray-triangle hit", color="#8C2D19", size=7)
add_points(fig, tri, "triangle vertices", color="#5C3A00", size=4)
fig.update_layout(title="Line-plane solve and ray-triangle domain filter", scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"}, legend={"orientation": "h", "y": -0.05}, margin={"l": 0, "r": 0, "t": 50, "b": 0})
linear_path = save_plotly_html(fig, TOPIC, "interactive", "line-plane-ray-triangle-parameters.html")
artifact_paths.append(linear_path)

check_values["line_plane"] = {"status": line_hit["status"], "t": float(line_hit["t"]), "denominator": float(line_hit["denominator"]), "plane_residual": abs(float(line_hit["residual"]))}
check_values["ray_triangle"] = {
    "hit": bool(ray_hit["hit"]),
    "t": float(ray_hit["t"]),
    "barycentric": np.asarray(ray_hit["barycentric"]).round(8).tolist(),
    "barycentric_sum_error": abs(float(np.sum(ray_hit["barycentric"]) - 1.0)),
    "point_residual": float(ray_hit["point_residual"]),
}
display_artifact(linear_path, width="100%", height=560)
check_values["line_plane"], check_values["ray_triangle"]


## Polygon and Polyhedron Intersections: Half-Spaces, Edges, and Slabs

A convex polyhedron can be read two ways. As a mesh, it has vertices, edges, and faces. As a solid, it is an intersection of half-spaces. Plane slicing uses the mesh view: every crossed edge contributes a point, and the points are sorted in the section plane. Ray-box intersection uses the half-space view: each pair of parallel faces clips the allowed interval for `t`.

The rotatable artifact combines both views. The amber polygon is the plane section of a box mesh. The red segment is exactly the subinterval of the ray that survives all three slabs of the same AABB.


In [ ]:
def unique_rows(points: list[np.ndarray], *, decimals: int = 10) -> np.ndarray:
    if not points:
        return np.empty((0, 3))
    keyed: dict[tuple[float, float, float], np.ndarray] = {}
    for p in points:
        keyed[tuple(np.round(np.asarray(p, dtype=float), decimals))] = np.asarray(p, dtype=float)
    return np.vstack(list(keyed.values()))


def section_polygon_from_edges(vertices: np.ndarray, edges: np.ndarray, normal: np.ndarray, c: float, *, eps: float = 1e-10) -> np.ndarray:
    n = normalize(normal)
    points: list[np.ndarray] = []
    signed = vertices @ n - c
    for i, j in edges:
        a, b = vertices[i], vertices[j]
        da, db = signed[i], signed[j]
        if abs(da) <= eps:
            points.append(a)
        if abs(db) <= eps:
            points.append(b)
        if da * db < -eps:
            alpha = da / (da - db)
            points.append(a + alpha * (b - a))
    pts = unique_rows(points)
    if len(pts) <= 2:
        return pts
    u, v = plane_basis(n)
    center = pts.mean(axis=0)
    angles = np.arctan2((pts - center) @ v, (pts - center) @ u)
    return pts[np.argsort(angles)]


def ray_aabb_slab(O: np.ndarray, d: np.ndarray, bounds: np.ndarray, *, eps: float = 1e-12) -> dict[str, object]:
    O = np.asarray(O, dtype=float)
    d = normalize(d)
    lo, hi = np.asarray(bounds[0], dtype=float), np.asarray(bounds[1], dtype=float)
    t_enter = -math.inf
    t_exit = math.inf
    intervals = []
    enter_axis = exit_axis = None
    for axis, name in enumerate("xyz"):
        if abs(d[axis]) <= eps:
            if O[axis] < lo[axis] or O[axis] > hi[axis]:
                return {"hit": False, "reason": f"parallel outside {name}-slab", "intervals": intervals}
            intervals.append({"axis": name, "near": -math.inf, "far": math.inf})
            continue
        t0 = (lo[axis] - O[axis]) / d[axis]
        t1 = (hi[axis] - O[axis]) / d[axis]
        near, far = sorted([float(t0), float(t1)])
        intervals.append({"axis": name, "near": near, "far": far})
        if near > t_enter:
            t_enter = near
            enter_axis = name
        if far < t_exit:
            t_exit = far
            exit_axis = name
        if t_enter > t_exit:
            return {"hit": False, "reason": "empty clipped interval", "intervals": intervals, "t_enter": t_enter, "t_exit": t_exit}
    if t_exit < 0:
        return {"hit": False, "reason": "box is behind ray", "intervals": intervals, "t_enter": t_enter, "t_exit": t_exit}
    t_enter = max(t_enter, 0.0)
    return {"hit": True, "t_enter": float(t_enter), "t_exit": float(t_exit), "enter_axis": enter_axis, "exit_axis": exit_axis, "intervals": intervals}


box_mesh = trimesh.creation.box(extents=(2.0, 1.35, 1.15))
box_vertices = np.asarray(box_mesh.vertices)
box_faces = np.asarray(box_mesh.faces)
box_edges = np.array([
    (i, j)
    for i in range(len(box_vertices))
    for j in range(i + 1, len(box_vertices))
    if np.count_nonzero(np.abs(box_vertices[i] - box_vertices[j]) > 1e-9) == 1
], dtype=int)
section_n = normalize(np.array([0.55, -0.25, 1.0]))
section_c = 0.08
section_pts = section_polygon_from_edges(box_vertices, box_edges, section_n, section_c)

ray_box_O = np.array([-1.85, -0.45, 0.42])
ray_box_d = normalize(np.array([1.0, 0.24, -0.16]))
slab = ray_aabb_slab(ray_box_O, ray_box_d, box_mesh.bounds)
entry_pt = ray_box_O + slab["t_enter"] * ray_box_d
exit_pt = ray_box_O + slab["t_exit"] * ray_box_d

fig = go.Figure()
fig.add_trace(go.Mesh3d(x=box_vertices[:, 0], y=box_vertices[:, 1], z=box_vertices[:, 2], i=box_faces[:, 0], j=box_faces[:, 1], k=box_faces[:, 2], color="#7AA6C2", opacity=0.28, name="box mesh / convex polyhedron"))
closed = np.vstack([section_pts, section_pts[0]])
fig.add_trace(go.Mesh3d(x=section_pts[:, 0], y=section_pts[:, 1], z=section_pts[:, 2], i=[0] * max(len(section_pts) - 2, 0), j=list(range(1, max(len(section_pts) - 1, 1))), k=list(range(2, len(section_pts))), color="#E3A018", opacity=0.62, name="plane-section polygon"))
add_polyline(fig, closed, "section polygon boundary", color="#8A5A00", width=6)
ray_plot_ts = np.linspace(0, slab["t_exit"] + 0.75, 50)
ray_plot = ray_box_O[None, :] + ray_plot_ts[:, None] * ray_box_d[None, :]
add_polyline(fig, ray_plot, "ray before slab clipping", color="#C44E52", width=3)
add_polyline(fig, np.vstack([entry_pt, exit_pt]), "accepted slab interval", color="#B00020", width=8)
add_points(fig, np.vstack([entry_pt, exit_pt]), "entry / exit", color="#B00020", size=6)
fig.update_layout(title="Plane section polygon and AABB slab interval", scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"}, legend={"orientation": "h", "y": -0.05}, margin={"l": 0, "r": 0, "t": 50, "b": 0})
poly_path = save_plotly_html(fig, TOPIC, "interactive", "polyhedron-section-slabs.html")
artifact_paths.append(poly_path)

section_residuals = section_pts @ section_n - section_c
inside_entry = np.all(entry_pt >= box_mesh.bounds[0] - 1e-9) and np.all(entry_pt <= box_mesh.bounds[1] + 1e-9)
inside_exit = np.all(exit_pt >= box_mesh.bounds[0] - 1e-9) and np.all(exit_pt <= box_mesh.bounds[1] + 1e-9)
check_values["polyhedron_section"] = {"vertex_count": int(len(section_pts)), "max_plane_residual": float(np.max(np.abs(section_residuals))), "edge_count_checked": int(len(box_edges))}
check_values["slab_box"] = {"hit": bool(slab["hit"]), "t_enter": float(slab["t_enter"]), "t_exit": float(slab["t_exit"]), "enter_axis": slab["enter_axis"], "exit_axis": slab["exit_axis"], "entry_inside": bool(inside_entry), "exit_inside": bool(inside_exit)}
display_artifact(poly_path, width="100%", height=560)
check_values["polyhedron_section"], check_values["slab_box"]


## Quadrics and Higher Implicit Surfaces: Substitute the Line, Then Classify Roots

For a general implicit surface `F(X) = 0`, a linear component changes the geometric question into a one-variable equation `F(P + t d) = 0`. A quadric gives a quadratic polynomial, so the discriminant distinguishes no hit, tangency, and two crossings. A standard cylinder and cone are also quadratic after transforming the line into the object's local frame. A torus gives a quartic, so root filtering and residual checks become especially important.

The diagnostic plot intentionally shows residual curves, not just intersection points. This makes tangency and missed roots visible: a root is trustworthy only if the residual returns to zero and the domain filter accepts the point.


In [ ]:
def real_roots(poly: Polynomial, *, imag_tol: float = 1e-7) -> list[float]:
    coef = np.trim_zeros(np.asarray(poly.coef, dtype=float), "b")
    if len(coef) <= 1:
        return []
    roots = np.roots(coef[::-1])
    return sorted(float(r.real) for r in roots if abs(r.imag) <= imag_tol)


def line_polynomials(P: np.ndarray, d: np.ndarray) -> tuple[Polynomial, Polynomial, Polynomial]:
    P = np.asarray(P, dtype=float)
    d = normalize(d)
    return Polynomial([P[0], d[0]]), Polynomial([P[1], d[1]]), Polynomial([P[2], d[2]])


def sphere_poly(P: np.ndarray, d: np.ndarray, r: float = 1.0) -> Polynomial:
    X, Y, Z = line_polynomials(P, d)
    return X * X + Y * Y + Z * Z - r * r


def cylinder_poly(P: np.ndarray, d: np.ndarray, r: float = 0.75) -> Polynomial:
    X, Y, _ = line_polynomials(P, d)
    return X * X + Y * Y - r * r


def cone_poly(P: np.ndarray, d: np.ndarray, k: float = 0.55) -> Polynomial:
    X, Y, Z = line_polynomials(P, d)
    return X * X + Y * Y - (k * k) * Z * Z


def torus_poly(P: np.ndarray, d: np.ndarray, R: float = 1.2, r: float = 0.35) -> Polynomial:
    X, Y, Z = line_polynomials(P, d)
    squared_radius = X * X + Y * Y + Z * Z
    ring = squared_radius + (R * R - r * r)
    return ring * ring - 4 * R * R * (X * X + Y * Y)


implicit_queries = {
    "sphere": {"P": np.array([-1.7, -0.25, 0.25]), "d": normalize(np.array([1.0, 0.2, -0.05])), "poly": sphere_poly, "domain": lambda Q: True, "surface": {"r": 1.0}},
    "finite cylinder": {"P": np.array([-1.4, 0.45, -0.4]), "d": normalize(np.array([1.0, -0.2, 0.15])), "poly": cylinder_poly, "domain": lambda Q: -0.9 <= Q[2] <= 0.9, "surface": {"r": 0.75, "zmin": -0.9, "zmax": 0.9}},
    "finite cone": {"P": np.array([-1.2, 0.25, 1.3]), "d": normalize(np.array([1.0, 0.05, -0.4])), "poly": cone_poly, "domain": lambda Q: 0.0 <= Q[2] <= 1.6, "surface": {"k": 0.55, "zmin": 0.0, "zmax": 1.6}},
    "torus": {"P": np.array([-2.2, 0.35, 0.18]), "d": normalize(np.array([1.0, 0.0, 0.0])), "poly": torus_poly, "domain": lambda Q: True, "surface": {"R": 1.2, "r": 0.35}},
}

root_data: dict[str, dict[str, object]] = {}
fig, axes = plt.subplots(2, 2, figsize=(11, 7.2), sharex=True)
t_values = np.linspace(0.0, 4.1, 500)
for ax, (name, query) in zip(axes.ravel(), implicit_queries.items()):
    P, d = query["P"], query["d"]
    poly = query["poly"](P, d)
    values = poly(t_values)
    scale = max(float(np.max(np.abs(values))), 1.0)
    roots = real_roots(poly)
    accepted = []
    residuals = []
    for tval in roots:
        Q = P + tval * d
        residual = float(poly(tval))
        residuals.append(abs(residual))
        if tval >= -1e-9 and query["domain"](Q):
            accepted.append(float(tval))
    ax.plot(t_values, values / scale, color="#244C5A", lw=2)
    ax.axhline(0, color="#111827", lw=1)
    for tval in accepted:
        ax.plot([tval], [0], marker="o", color="#B00020", ms=6)
        ax.text(tval, 0.08, f"t={tval:.2f}", ha="center", fontsize=8)
    ax.set_title(f"{name}: degree {poly.degree()}, accepted roots {len(accepted)}", loc="left", fontsize=10, weight="bold")
    ax.set_ylabel("scaled F(P+t d)")
    ax.grid(True, alpha=0.25)
    root_data[name] = {"degree": int(poly.degree()), "all_real_roots": [float(tval) for tval in roots], "accepted_nonnegative_roots": accepted, "max_root_residual": float(max(residuals) if residuals else 0.0)}
axes[-1, 0].set_xlabel("line parameter t")
axes[-1, 1].set_xlabel("line parameter t")
fig.suptitle("Implicit-surface intersection as one-variable root finding", x=0.02, ha="left", fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
implicit_diag_path = save_matplotlib(fig, TOPIC, "figures", "implicit-surface-root-diagnostics.png")
plt.close(fig)
artifact_paths.append(implicit_diag_path)
check_values["implicit_roots"] = root_data
display_artifact(implicit_diag_path, width=900)
root_data


In [ ]:
def add_surface_scene(fig: go.Figure, row: int, col: int, name: str, query: dict[str, object]) -> None:
    P, d = query["P"], query["d"]
    roots = root_data[name]["accepted_nonnegative_roots"]
    surface = query["surface"]
    if name == "sphere":
        u = np.linspace(0, 2 * np.pi, 36)
        v = np.linspace(0, np.pi, 18)
        uu, vv = np.meshgrid(u, v)
        r0 = surface["r"]
        x = r0 * np.cos(uu) * np.sin(vv)
        y = r0 * np.sin(uu) * np.sin(vv)
        z = r0 * np.cos(vv)
    elif name == "finite cylinder":
        theta = np.linspace(0, 2 * np.pi, 36)
        zvals = np.linspace(surface["zmin"], surface["zmax"], 18)
        tt, zz = np.meshgrid(theta, zvals)
        x = surface["r"] * np.cos(tt)
        y = surface["r"] * np.sin(tt)
        z = zz
    elif name == "finite cone":
        theta = np.linspace(0, 2 * np.pi, 36)
        zvals = np.linspace(surface["zmin"], surface["zmax"], 18)
        tt, zz = np.meshgrid(theta, zvals)
        x = surface["k"] * zz * np.cos(tt)
        y = surface["k"] * zz * np.sin(tt)
        z = zz
    else:
        u = np.linspace(0, 2 * np.pi, 50)
        v = np.linspace(0, 2 * np.pi, 22)
        uu, vv = np.meshgrid(u, v)
        R0, r0 = surface["R"], surface["r"]
        x = (R0 + r0 * np.cos(vv)) * np.cos(uu)
        y = (R0 + r0 * np.cos(vv)) * np.sin(uu)
        z = r0 * np.sin(vv)
    fig.add_trace(go.Surface(x=x, y=y, z=z, opacity=0.48, showscale=False, name=name, colorscale=[[0, "#8FB9A8"], [1, "#8FB9A8"]]), row=row, col=col)
    ts = np.linspace(0, max([4.1, *roots]), 80)
    line = P[None, :] + ts[:, None] * d[None, :]
    fig.add_trace(go.Scatter3d(x=line[:, 0], y=line[:, 1], z=line[:, 2], mode="lines", name=f"{name} line", line={"color": "#B00020", "width": 5}, showlegend=False), row=row, col=col)
    if roots:
        pts = np.vstack([P + tval * d for tval in roots])
        fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=f"{name} roots", marker={"size": 4, "color": "#111827"}, showlegend=False), row=row, col=col)


fig = make_subplots(rows=2, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}], [{"type": "scene"}, {"type": "scene"}]], subplot_titles=["sphere", "finite cylinder", "finite cone", "torus"])
for (row, col), name in zip([(1, 1), (1, 2), (2, 1), (2, 2)], implicit_queries):
    add_surface_scene(fig, row, col, name, implicit_queries[name])
fig.update_layout(title="Implicit-surface roots as visible crossings", height=760, margin={"l": 0, "r": 0, "t": 70, "b": 0})
for scene_name in ["scene", "scene2", "scene3", "scene4"]:
    fig.layout[scene_name].update(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z")
implicit_atlas_path = save_plotly_html(fig, TOPIC, "interactive", "implicit-surface-root-atlas.html")
artifact_paths.append(implicit_atlas_path)
display_artifact(implicit_atlas_path, width="100%", height=760)


### Symbolic Degree Scaffold

The same substitution explains why the algorithms have different numerical character. A sphere or cone gives degree two in `t`; a torus gives degree four. That degree check is a small proof scaffold: before choosing a root solver, the code confirms the polynomial class created by the geometry.


In [ ]:
t = sp.symbols("t")
px, py, pz, dx, dy, dz, R, r, k = sp.symbols("p_x p_y p_z d_x d_y d_z R r k")
x, y, z = px + t * dx, py + t * dy, pz + t * dz
sphere_degree = sp.Poly(x**2 + y**2 + z**2 - r**2, t).degree()
cone_degree = sp.Poly(x**2 + y**2 - k**2 * z**2, t).degree()
torus_expr = (x**2 + y**2 + z**2 + R**2 - r**2) ** 2 - 4 * R**2 * (x**2 + y**2)
torus_degree = sp.Poly(sp.expand(torus_expr), t).degree()
check_values["symbolic_degrees"] = {"sphere": int(sphere_degree), "cone": int(cone_degree), "torus": int(torus_degree)}
check_values["symbolic_degrees"]


## Separating-Axis Diagnostics for Oriented Boxes

The method of separating axes turns a 3D convex intersection test into projection interval comparisons. For two oriented boxes, the candidate axes are the three axes of each box plus the nine cross products of box edges. If any candidate has a positive gap between the projected intervals, the boxes are disjoint. If every candidate overlaps, the SAT says the boxes intersect.

The diagnostic below records the signed gap for each axis. Positive means a separating axis was found. Negative means the intervals overlap, and the magnitude is the overlap depth along that axis. This is more useful than a Boolean because it identifies the exact direction responsible for rejection.


In [ ]:
def rotation_z(angle: float) -> np.ndarray:
    c, s = math.cos(angle), math.sin(angle)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def rotation_y(angle: float) -> np.ndarray:
    c, s = math.cos(angle), math.sin(angle)
    return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])


def oriented_box(center: np.ndarray, axes: np.ndarray, half: np.ndarray) -> dict[str, np.ndarray]:
    axes = np.asarray(axes, dtype=float)
    assert np.allclose(axes.T @ axes, np.eye(3), atol=1e-10)
    return {"center": np.asarray(center, dtype=float), "axes": axes, "half": np.asarray(half, dtype=float)}


def obb_vertices(box: dict[str, np.ndarray]) -> np.ndarray:
    signs = np.array([[sx, sy, sz] for sx in [-1, 1] for sy in [-1, 1] for sz in [-1, 1]], dtype=float)
    return np.vstack([box["center"] + box["axes"] @ (sign * box["half"]) for sign in signs])


def project_box(box: dict[str, np.ndarray], axis: np.ndarray) -> tuple[float, float]:
    axis = normalize(axis)
    center_projection = float(np.dot(box["center"], axis))
    radius = sum(float(box["half"][i] * abs(np.dot(box["axes"][:, i], axis))) for i in range(3))
    return center_projection - radius, center_projection + radius


def interval_signed_gap(a: tuple[float, float], b: tuple[float, float]) -> float:
    if a[1] < b[0]:
        return float(b[0] - a[1])
    if b[1] < a[0]:
        return float(a[0] - b[1])
    return -float(min(a[1], b[1]) - max(a[0], b[0]))


def sat_obb(A: dict[str, np.ndarray], B: dict[str, np.ndarray], *, eps: float = 1e-10) -> dict[str, object]:
    candidates: list[tuple[str, np.ndarray]] = []
    for i in range(3):
        candidates.append((f"A{i}", A["axes"][:, i]))
    for i in range(3):
        candidates.append((f"B{i}", B["axes"][:, i]))
    for i in range(3):
        for j in range(3):
            axis = np.cross(A["axes"][:, i], B["axes"][:, j])
            if np.linalg.norm(axis) > eps:
                candidates.append((f"A{i}xB{j}", axis))
    rows = []
    for label, axis in candidates:
        axis = normalize(axis)
        interval_A = project_box(A, axis)
        interval_B = project_box(B, axis)
        gap = interval_signed_gap(interval_A, interval_B)
        rows.append({"axis": label, "gap": gap, "interval_A": interval_A, "interval_B": interval_B, "direction": axis})
    rows.sort(key=lambda item: item["gap"], reverse=True)
    separating = [row for row in rows if row["gap"] > eps]
    return {"intersects": len(separating) == 0, "separating_axes": separating, "axes": rows}


A = oriented_box(np.array([0.0, 0.0, 0.0]), np.eye(3), np.array([1.0, 0.55, 0.45]))
rot_B = rotation_z(math.radians(28.0)) @ rotation_y(math.radians(10.0))
B_separated = oriented_box(np.array([2.10, 0.35, 0.05]), rot_B, np.array([0.75, 0.45, 0.55]))
B_overlap = oriented_box(np.array([1.70, 0.35, 0.05]), rot_B, np.array([0.75, 0.45, 0.55]))

sat_sep = sat_obb(A, B_separated)
sat_overlap = sat_obb(A, B_overlap)
sat_swap = sat_obb(B_separated, A)

top_axes = sat_sep["axes"][:15]
labels = [row["axis"] for row in top_axes]
gaps = np.array([row["gap"] for row in top_axes])
colors = ["#B00020" if gap > 0 else "#5F8D4E" for gap in gaps]
fig, ax = plt.subplots(figsize=(10.5, 5.5))
y = np.arange(len(labels))
ax.barh(y, gaps, color=colors, alpha=0.88)
ax.axvline(0, color="#111827", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("signed projection gap (positive separates, negative overlaps)")
ax.set_title("SAT axis diagnostics for a separated pair of oriented boxes", loc="left", fontsize=13, weight="bold")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
sat_gap_path = save_matplotlib(fig, TOPIC, "figures", "sat-axis-gap-diagnostics.png")
plt.close(fig)
artifact_paths.append(sat_gap_path)

box_faces = np.array([[0, 1, 3], [0, 3, 2], [4, 6, 7], [4, 7, 5], [0, 4, 5], [0, 5, 1], [2, 3, 7], [2, 7, 6], [0, 2, 6], [0, 6, 4], [1, 5, 7], [1, 7, 3]])

def add_box(fig: go.Figure, box: dict[str, np.ndarray], name: str, color: str, opacity: float) -> None:
    verts = obb_vertices(box)
    fig.add_trace(go.Mesh3d(x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=box_faces[:, 0], j=box_faces[:, 1], k=box_faces[:, 2], name=name, color=color, opacity=opacity))

fig3 = go.Figure()
add_box(fig3, A, "box A", "#6BA292", 0.42)
add_box(fig3, B_separated, "box B separated", "#D88C60", 0.46)
best_axis = sat_sep["separating_axes"][0]
axis = best_axis["direction"]
add_polyline(fig3, np.vstack([A["center"] - 1.2 * axis, A["center"] + 3.0 * axis]), f"separating axis {best_axis['axis']}", color="#B00020", width=7)
fig3.update_layout(title="Oriented boxes and a discovered separating axis", scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"}, legend={"orientation": "h", "y": -0.05}, margin={"l": 0, "r": 0, "t": 50, "b": 0})
sat_scene_path = save_plotly_html(fig3, TOPIC, "interactive", "sat-oriented-boxes.html")
artifact_paths.append(sat_scene_path)

check_values["sat"] = {
    "separated_intersects": bool(sat_sep["intersects"]),
    "overlap_intersects": bool(sat_overlap["intersects"]),
    "separating_axis": sat_sep["separating_axes"][0]["axis"],
    "separating_gap": float(sat_sep["separating_axes"][0]["gap"]),
    "swap_intersects_matches": bool(sat_sep["intersects"] == sat_swap["intersects"]),
    "tested_axis_count": int(len(sat_sep["axes"])),
}
display_artifact(sat_gap_path, width=900)
display_artifact(sat_scene_path, width="100%", height=560)
check_values["sat"]


## Applied Lab: Turn a Boolean Into a Diagnostic

A production graphics pipeline often asks only whether two objects intersect. For debugging, that is too little. Use the cells above as a lab template:

1. Pick a primitive pair and write down the representation and domain before solving.
2. Return the certificate with the Boolean: `t`, barycentric weights, plane residuals, slab interval, root residuals, or SAT gaps.
3. Perturb one input toward a degenerate case: near-parallel line/plane, ray grazing a triangle edge, slab direction component near zero, tangent quadric line, or almost-touching boxes.
4. Plot the certificate as the input changes. A stable algorithm should change continuously until a genuine topological event occurs.

The small sweep below varies a ray direction component near a slab-parallel case. The accepted interval stays meaningful until the ray no longer reaches the box. This is the sort of diagnostic that catches tolerance bugs earlier than a rendered frame does.


In [ ]:
eps_values = np.linspace(-0.18, 0.18, 81)
interval_lengths = []
entry_values = []
valid_flags = []
for eps in eps_values:
    d = normalize(np.array([1.0, eps, -0.16]))
    result = ray_aabb_slab(ray_box_O, d, box_mesh.bounds)
    valid_flags.append(bool(result["hit"]))
    if result["hit"]:
        entry_values.append(float(result["t_enter"]))
        interval_lengths.append(float(result["t_exit"] - result["t_enter"]))
    else:
        entry_values.append(np.nan)
        interval_lengths.append(np.nan)

fig, ax = plt.subplots(figsize=(9.5, 4.7))
ax.plot(eps_values, interval_lengths, color="#244C5A", lw=2, label="accepted interval length")
ax.plot(eps_values, entry_values, color="#D88C60", lw=2, label="entry parameter")
ax.axvline(ray_box_d[1] / ray_box_d[0], color="#111827", ls="--", lw=1, label="chapter example direction ratio")
ax.set_xlabel("y component before normalization")
ax.set_ylabel("parameter value")
ax.set_title("Slab diagnostic sweep near a direction perturbation", loc="left", fontsize=13, weight="bold")
ax.grid(True, alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
slab_sweep_path = save_matplotlib(fig, TOPIC, "figures", "slab-interval-perturbation-sweep.png")
plt.close(fig)
artifact_paths.append(slab_sweep_path)
check_values["slab_sweep"] = {"sample_count": int(len(eps_values)), "valid_count": int(np.count_nonzero(valid_flags)), "min_interval_length": float(np.nanmin(interval_lengths)), "max_interval_length": float(np.nanmax(interval_lengths))}
display_artifact(slab_sweep_path, width=860)
check_values["slab_sweep"]


## Final Sanity Checks

The final cell asserts the certificates that matter for this chapter:

- line-plane and ray-triangle reported points satisfy their equations and domains;
- the polyhedron section vertices lie in the slicing plane;
- the slab interval is nonempty and its endpoints lie inside the box;
- implicit-surface roots have small residuals and the symbolic degrees match the expected algorithms;
- SAT detects the separated and overlapping box examples and is invariant under object order;
- every saved artifact exists and is nonempty.


In [ ]:
assert check_values["storyboard_nodes"] >= 12
assert check_values["line_plane"]["status"] == "point"
assert check_values["line_plane"]["plane_residual"] < 1e-12
assert check_values["ray_triangle"]["hit"] is True
assert check_values["ray_triangle"]["barycentric_sum_error"] < 1e-12
assert min(check_values["ray_triangle"]["barycentric"]) >= -1e-12
assert check_values["ray_triangle"]["point_residual"] < 1e-12
assert check_values["polyhedron_section"]["vertex_count"] >= 3
assert check_values["polyhedron_section"]["max_plane_residual"] < 1e-10
assert check_values["slab_box"]["hit"] is True
assert check_values["slab_box"]["t_enter"] <= check_values["slab_box"]["t_exit"]
assert check_values["slab_box"]["entry_inside"] and check_values["slab_box"]["exit_inside"]
for name, data in check_values["implicit_roots"].items():
    assert data["accepted_nonnegative_roots"], name
    assert data["max_root_residual"] < 1e-7, (name, data)
assert check_values["symbolic_degrees"] == {"sphere": 2, "cone": 2, "torus": 4}
assert check_values["sat"]["separated_intersects"] is False
assert check_values["sat"]["overlap_intersects"] is True
assert check_values["sat"]["swap_intersects_matches"] is True
assert check_values["sat"]["separating_gap"] > 0
assert check_values["slab_sweep"]["valid_count"] > 0
assert check_values["slab_sweep"]["min_interval_length"] > 0

require_nonempty(artifact_paths, min_bytes=1500)
invariants_path = save_json(check_values, TOPIC, "checks", "intersection-invariants.json")
final_sanity = {
    "topic": TOPIC,
    "title": ENTRY_TITLE,
    "source_span": SOURCE_SPAN,
    "artifacts": artifact_report(artifact_paths + [invariants_path], root=BOOK_ROOT),
    "checks": {
        "line_plane_residual": check_values["line_plane"]["plane_residual"],
        "ray_triangle_barycentric": check_values["ray_triangle"],
        "polyhedron_section": check_values["polyhedron_section"],
        "slab_box": check_values["slab_box"],
        "implicit_root_families": list(check_values["implicit_roots"].keys()),
        "symbolic_degrees": check_values["symbolic_degrees"],
        "sat": check_values["sat"],
        "slab_sweep": check_values["slab_sweep"],
    },
    "libraries": {
        "plotly": "rotatable 3D scenes for spatial intersection state",
        "trimesh": "box mesh vertices, faces, and unique edges for polyhedron slicing",
        "sympy": "symbolic degree checks after line substitution",
        "networkx": "algorithm dependency storyboard",
        "matplotlib": "durable static diagnostic plots",
        "numpy": "linear algebra, polynomial coefficients, interval and SAT arithmetic",
    },
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
require_nonempty([invariants_path, sanity_path], min_bytes=500)
final_sanity


## Takeaways

- Intersections in 3D are best treated as **solve, restrict, classify, and certify**.
- A line-plane denominator, a ray-triangle barycentric triple, a slab interval, a polynomial residual, and a SAT projection gap are all certificates, not incidental implementation details.
- Convex polyhedra invite two complementary views: mesh edges for slicing and half-spaces for clipping.
- Quadrics, cylinders, cones, and tori are unified by line substitution; the degree of the resulting polynomial tells you what numerical tool is needed.
- A notebook-quality intersection routine should expose enough diagnostic state to make degeneracies inspectable before they become rendering or collision bugs.
